# Kinetica Langchain extension demo

This notebook provides an example usage for the `langchain_kinetica` package.

## Configuration

You must configure the Kinetica connection and the LLM context. The LLM context is an object stored in the database that provides information needed for inferencing. 

See the documentation site for more information about the [LLM Context](https://docs.kinetica.com/7.1/sql-gpt/concepts/#sql-gpt-context)

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_kinetica import KineticaChatLLM, KineticaSqlOutputParser, SqlResponse

# Get DB connection parameters from environment variables.
k_url = os.environ["KINETICA_URL"]
k_login = os.environ["KINETICA_LOGIN"]
k_passwd = os.environ["KINETICA_PASSWORD"]

# create the Kinetica connection
kdbc = KineticaChatLLM._create_kdbc(host=k_url, login=k_login, password=k_passwd)

# create the Kinetica LLM
kinetica_llm = KineticaChatLLM(kdbc=kdbc)

# Set the LLM context name. 
kinetica_ctx = 'telecom.chad_test'

In [ ]:
# load the context from the database
ctx_messages = kinetica_llm.load_messages_from_context(kinetica_ctx)

# Add the input prompt. This is where input question will be substituted.
ctx_messages.append(("human", "{input}"))

# Create the prompt template.
prompt_template = ChatPromptTemplate.from_messages(ctx_messages)
prompt_template.pretty_print()

# create the chain. 
# note: The KineticaSqlOutputParser will execute the SQL statement and is optional.
chain = prompt_template | kinetica_llm | KineticaSqlOutputParser(kdbc=kdbc)

In [ ]:
# Test the inferencing

# Here you must ask a question relevant to the LLM context provided in the prompt template.
response: SqlResponse = chain.invoke({"input": "What is the invoice date and price of transactions made by customer C274529?"})

print(f"SQL: {response.sql}")
response.dataframe